# Urban Heat Island (UHI) Analysis — v3
## MSF · Tondo, Manila · Z_GIS Salzburg

---

### v3 features
- **Sentinel-3 SLSTR LST** via Copernicus Data Space API (primary)
- **MODIS Terra+Aqua** auto-fallback (GEE)
- Landsat 8+9 combined (~8-day, 30 m)
- Regression-based fusion (coarse → 30 m)
- Monthly time-series + validation
- Resolution-aware stack

> Edit **CONFIGURATION** only. Set `THERMAL_SOURCE = "S3"` or `"MODIS"`.

---

## 0. Setup

### GEE
1. Register: [code.earthengine.google.com/register](https://code.earthengine.google.com/register)
2. Cloud project: [console.cloud.google.com](https://console.cloud.google.com)
3. Enable EE API

### Copernicus Data Space (free, for S3)
1. Register: [dataspace.copernicus.eu](https://dataspace.copernicus.eu)
2. Use your email + password below

---

In [1]:
# 0A. INSTALL
!pip install earthengine-api folium pandas matplotlib seaborn geopandas xarray netCDF4 requests planetary-computer pystac-client -q
print("Installed.")

Installed.


In [3]:
# 0B. AUTHENTICATE
GEE_PROJECT = "uhi-msf-analysis"

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print(f"GEE: {GEE_PROJECT}")

import requests
CDSE_USERNAME = "**********"
CDSE_PASSWORD = "**********"

def get_cdse_token():
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={"grant_type":"password","username":CDSE_USERNAME,"password":CDSE_PASSWORD,"client_id":"cdse-public"})
    if r.status_code == 200:
        print("CDSE authenticated.")
        return r.json()["access_token"]
    else:
        print(f"CDSE auth failed: {r.status_code}")
        return None

cdse_token = get_cdse_token()

GEE: uhi-msf-analysis
CDSE authenticated.


In [4]:
# 0C. IMPORTS
import folium, pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.colors as mcolors, seaborn as sns
import xarray as xr, json, os, zipfile, io, tempfile, glob, warnings
warnings.filterwarnings("ignore")
from datetime import datetime, timedelta

def add_ee_layer(m, img, vis, name, shown=True):
    mid = img.getMapId(vis)
    folium.TileLayer(tiles=mid["tile_fetcher"].url_format, attr="GEE", name=name,
                     overlay=True, control=True, show=shown).add_to(m)

def make_map(lat, lon, zoom=14):
    return folium.Map(location=[lat, lon], zoom_start=zoom, tiles="OpenStreetMap")

print("Libraries loaded.")

Libraries loaded.


---
## 1. CONFIGURATION
---

In [5]:
# 1. CONFIG
AOI_BBOX = [120.950, 14.600, 120.985, 14.640]
# AOI_ASSET = "FAO/GAUL/2015/level2"
# AOI_FILTER = ee.Filter.eq("ADM2_NAME", "Manila")

YEAR_START = 2019
YEAR_END = 2025
DRY_MONTHS = [12,1,2,3,4,5]
WET_MONTHS = [6,7,8,9,10,11]
THERMAL_SOURCE = "S3"  # "S3" or "MODIS"
MAX_CLOUD_COVER = 30
UHI_BUFFER_M = 5000
W_LST=0.30; W_POP=0.25; W_BUILD=0.20; W_NDVI=0.15; W_HEALTH=0.10
PROJECT_NAME = "UHI_Tondo_Manila"
assert abs(W_LST+W_POP+W_BUILD+W_NDVI+W_HEALTH-1.0)<0.001
print(f"Config: {PROJECT_NAME} | {AOI_BBOX} | {YEAR_START}-{YEAR_END} | Thermal: {THERMAL_SOURCE}")

Config: UHI_Tondo_Manila | [120.95, 14.6, 120.985, 14.64] | 2019-2025 | Thermal: S3


## 2. Area of Interest

In [6]:
# 2. AOI
try:
    aoi = ee.FeatureCollection(AOI_ASSET).filter(AOI_FILTER).geometry()
except NameError:
    aoi = ee.Geometry.Rectangle(AOI_BBOX)
aoi_buffer = aoi.buffer(UHI_BUFFER_M)
aoi_ring = aoi_buffer.difference(aoi)
center_lat = (AOI_BBOX[1]+AOI_BBOX[3])/2
center_lon = (AOI_BBOX[0]+AOI_BBOX[2])/2
m = make_map(center_lat, center_lon, 13)
folium.GeoJson(aoi.getInfo(), name="AOI", style_function=lambda x:{"color":"red","weight":2,"fillOpacity":0.1}).add_to(m)
folium.GeoJson(aoi_ring.getInfo(), name="Ring", style_function=lambda x:{"color":"blue","weight":1,"fillOpacity":0.05}).add_to(m)
folium.LayerControl().add_to(m)
m

## 3. Helper Functions

In [7]:
# 3. HELPERS
def mask_landsat_clouds(image):
    qa = image.select("QA_PIXEL")
    return image.updateMask(qa.bitwiseAnd(1<<3).eq(0)).updateMask(qa.bitwiseAnd(1<<4).eq(0))

def compute_lst_landsat(image):
    return image.addBands(image.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("LST"))

def compute_lst_modis(image):
    lst = image.select("LST_Day_1km").multiply(0.02).subtract(273.15).rename("LST_coarse")
    return image.addBands(lst).updateMask(image.select("QC_Day").bitwiseAnd(3).eq(0))

def compute_ndvi(img): return img.addBands(img.normalizedDifference(["SR_B5","SR_B4"]).rename("NDVI"))
def compute_ndbi(img): return img.addBands(img.normalizedDifference(["SR_B6","SR_B5"]).rename("NDBI"))
def compute_mndwi(img): return img.addBands(img.normalizedDifference(["SR_B3","SR_B6"]).rename("MNDWI"))

def filter_by_months(col, months):
    def tag(img):
        m = ee.Number(img.date().get("month"))
        return img.set("in_season", ee.List(months).contains(m))
    return col.map(tag).filter(ee.Filter.eq("in_season", True))

def add_time_props(img):
    d = img.date()
    return img.set("year",d.get("year")).set("month",d.get("month")).set("doy",d.getRelative("day","year"))

def min_max_normalize(image, geom, band_name=None, scale=30):
    if band_name: image = image.select(band_name)
    mm = image.reduceRegion(reducer=ee.Reducer.minMax(), geometry=geom, scale=scale, maxPixels=1e9)
    keys = mm.keys().getInfo()
    mn = [k for k in keys if "_min" in k][0]
    mx = [k for k in keys if "_max" in k][0]
    return image.subtract(ee.Number(mm.get(mn))).divide(ee.Number(mm.get(mx)).subtract(ee.Number(mm.get(mn)))).clamp(0,1)

print("Helpers defined.")

Helpers defined.


---
## 4. Landsat 8+9 Acquisition

In [8]:
# 4. LANDSAT 8+9
l9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi)
      .filterDate(f"{YEAR_START}-01-01",f"{YEAR_END}-12-31")
      .filter(ee.Filter.lt("CLOUD_COVER",MAX_CLOUD_COVER))
      .map(mask_landsat_clouds).map(compute_lst_landsat)
      .map(compute_ndvi).map(compute_ndbi).map(compute_mndwi)
      .map(add_time_props).map(lambda i:i.set("sensor","L9")))
l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi)
      .filterDate(f"{YEAR_START}-01-01",f"{YEAR_END}-12-31")
      .filter(ee.Filter.lt("CLOUD_COVER",MAX_CLOUD_COVER))
      .map(mask_landsat_clouds).map(compute_lst_landsat)
      .map(compute_ndvi).map(compute_ndbi).map(compute_mndwi)
      .map(add_time_props).map(lambda i:i.set("sensor","L8")))
landsat_col = l8.merge(l9).sort("system:time_start")
print(f"L8:{l8.size().getInfo()} L9:{l9.size().getInfo()} Total:{landsat_col.size().getInfo()}")
dates = landsat_col.aggregate_array("system:time_start").getInfo()
if dates:
    dts=[datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d") for d in dates]
    print(f"  {min(dts)} -> {max(dts)}")

L8:47 L9:27 Total:74
  2019-01-04 -> 2025-12-30


## 4B. Coarse Thermal: Sentinel-3 SLSTR (CDSE) or MODIS (GEE)

S3 SLSTR is **not in GEE**. Downloaded from Copernicus Data Space. MODIS auto-fallback.

In [ ]:
# 4B. SENTINEL-3 SLSTR (CDSE) + MODIS FALLBACK
coarse_source_used = None
s3_monthly_lst = {}

if THERMAL_SOURCE == "S3" and cdse_token:
    print("Downloading Sentinel-3 SLSTR LST from CDSE...")
    w,s,e,n = AOI_BBOX
    buf = UHI_BUFFER_M / 111000
    sw,ss,se,sn = w-buf, s-buf, e+buf, n+buf
    s3_scenes = 0
    active_token = cdse_token  # local copy we can refresh

    for yr in range(YEAR_START, YEAR_END+1):
        for mo in range(1,13):
            start_dt = f"{yr}-{mo:02d}-01T00:00:00.000Z"
            end_dt = f"{yr}-{mo+1:02d}-01T00:00:00.000Z" if mo<12 else f"{yr+1}-01-01T00:00:00.000Z"

            url = (
                "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
                f"?$filter=Collection/Name eq 'SENTINEL-3'"
                f" and Attributes/OData.CSC.StringAttribute/any("
                f"att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'SL_2_LST___')"
                f" and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON(("
                f"{sw} {ss},{se} {ss},{se} {sn},{sw} {sn},{sw} {ss}))')"
                f" and ContentDate/Start gt {start_dt}"
                f" and ContentDate/Start lt {end_dt}"
                f"&$top=50&$orderby=ContentDate/Start asc"
            )

            try:
                results = requests.get(url, timeout=30).json().get('value', [])
            except:
                continue
            if not results: continue

            lst_vals = []
            for prod in results[:10]:
                pid = prod['Id']
                try:
                    dl = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({pid})/$value"
                    hdr = {"Authorization": f"Bearer {active_token}"}
                    r = requests.get(dl, headers=hdr, timeout=120, stream=True)
                    if r.status_code == 401:
                        refreshed = get_cdse_token()
                        if refreshed:
                            active_token = refreshed
                            hdr = {"Authorization": f"Bearer {active_token}"}
                            r = requests.get(dl, headers=hdr, timeout=120, stream=True)
                    if r.status_code != 200: continue

                    with tempfile.TemporaryDirectory() as tmp:
                        zp = os.path.join(tmp, 'p.zip')
                        with open(zp, 'wb') as f:
                            for chunk in r.iter_content(8192): f.write(chunk)
                        with zipfile.ZipFile(zp) as zf:
                            lst_f = [x for x in zf.namelist() if 'LST_in.nc' in x]
                            if lst_f:
                                zf.extract(lst_f[0], tmp)
                                ds = xr.open_dataset(os.path.join(tmp, lst_f[0]))
                                if 'LST' in ds:
                                    v = ds['LST'].values.flatten()
                                    v = v[~np.isnan(v)]
                                    v = v[(v>200)&(v<350)] - 273.15
                                    if len(v)>0: lst_vals.extend(v.tolist())
                                ds.close()
                    s3_scenes += 1
                except:
                    continue

            if lst_vals:
                mean_v = np.nanmean(lst_vals)
                s3_monthly_lst[(yr,mo)] = ee.Image.constant(mean_v).rename('LST_coarse').clip(aoi_buffer).float()
                print(f"  {yr}-{mo:02d}: {len(results)} prods, mean={mean_v:.1f}C")

    if s3_monthly_lst:
        coarse_source_used = "S3"
        print(f"\nS3 SLSTR: {s3_scenes} products, {len(s3_monthly_lst)} months.")
    else:
        print("\nNo S3 data — falling back to MODIS.")

if coarse_source_used is None:
    if THERMAL_SOURCE=="S3": print("S3 unavailable — MODIS fallback.")
    mt = ee.ImageCollection('MODIS/061/MOD11A1').filterBounds(aoi).filterDate(f'{YEAR_START}-01-01',f'{YEAR_END}-12-31').map(compute_lst_modis)
    ma = ee.ImageCollection('MODIS/061/MYD11A1').filterBounds(aoi).filterDate(f'{YEAR_START}-01-01',f'{YEAR_END}-12-31').map(compute_lst_modis)
    modis_col = mt.merge(ma).sort('system:time_start').map(add_time_props)
    coarse_source_used = "MODIS"
    print(f"MODIS: {modis_col.size().getInfo()} scenes")

print(f"\nThermal source: {coarse_source_used}")

CDSE authenticated.
  2019-01: 50 prods, mean=-9.4C
CDSE authenticated.
  2019-02: 50 prods, mean=-13.8C
CDSE authenticated.
  2019-03: 50 prods, mean=-10.3C


In [9]:
# 4B. SENTINEL-3 SLSTR LST (Planetary Computer) + MODIS FALLBACK
# Reads directly from Azure cloud storage — no zip downloads!
import planetary_computer as pc
from pystac_client import Client as STACClient

coarse_source_used = None
s3_monthly_lst = {}

if THERMAL_SOURCE == "S3":
    print("Querying Sentinel-3 SLSTR LST on Planetary Computer...")

    try:
        catalog = STACClient.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=pc.sign_inplace
        )

        w, s, e, n = AOI_BBOX
        buf = UHI_BUFFER_M / 111000
        search_bbox = [w - buf, s - buf, e + buf, n + buf]

        for yr in range(YEAR_START, YEAR_END + 1):
            for mo in range(1, 13):
                end_mo = mo + 1 if mo < 12 else 1
                end_yr = yr if mo < 12 else yr + 1
                start_dt = f"{yr}-{mo:02d}-01"
                end_dt = f"{end_yr}-{end_mo:02d}-01"

                try:
                    search = catalog.search(
                        collections=["sentinel-3-slstr-lst-l2-netcdf"],
                        bbox=search_bbox,
                        datetime=f"{start_dt}/{end_dt}",
                        max_items=10
                    )
                    items = list(search.items())
                except Exception:
                    continue

                if not items:
                    continue

                lst_vals = []

                for item in items[:5]:  # 5 scenes per month max
                    try:
                        # Find the LST NetCDF asset
                        asset = None
                        for key in ['lst-in', 'LST_in', 'lst', 'safe-manifest']:
                            if key in item.assets:
                                asset = item.assets[key]
                                break
                        # Fallback: any asset with LST in the name
                        if asset is None:
                            for key, a in item.assets.items():
                                if 'lst' in key.lower() and key != 'tilejson' and key != 'rendered_preview':
                                    asset = a
                                    break

                        if asset is None:
                            continue

                        href = asset.href

                        # Read directly from cloud — no download!
                        ds = xr.open_dataset(href, engine='netcdf4')

                        # Find LST variable
                        lst_var = None
                        for vname in ds.data_vars:
                            if 'LST' in vname.upper() and 'flag' not in vname.lower():
                                lst_var = vname
                                break

                        if lst_var is None:
                            ds.close()
                            continue

                        # Get lat/lon for clipping
                        lat_var = lon_var = None
                        for coord in list(ds.coords) + list(ds.data_vars):
                            cl = coord.lower()
                            if 'lat' in cl and lat_var is None:
                                lat_var = coord
                            if 'lon' in cl and lon_var is None:
                                lon_var = coord

                        vals = ds[lst_var].values.flatten()

                        if lat_var and lon_var:
                            lats = ds[lat_var].values.flatten()
                            lons = ds[lon_var].values.flatten()
                            # Clip to AOI
                            mask = (
                                (lats >= search_bbox[1]) & (lats <= search_bbox[3]) &
                                (lons >= search_bbox[0]) & (lons <= search_bbox[2]) &
                                (~np.isnan(vals)) &
                                (vals > 270) & (vals < 350)
                            )
                            clipped = vals[mask]
                        else:
                            # No coords — filter by valid range only
                            clipped = vals[(~np.isnan(vals)) & (vals > 270) & (vals < 350)]

                        if len(clipped) > 0:
                            lst_vals.extend((clipped - 273.15).tolist())

                        ds.close()

                    except Exception:
                        continue

                if lst_vals:
                    mean_v = np.nanmean(lst_vals)
                    s3_monthly_lst[(yr, mo)] = (
                        ee.Image.constant(mean_v)
                        .rename('LST_coarse')
                        .clip(aoi_buffer)
                        .float()
                    )
                    print(f"  {yr}-{mo:02d}: {len(items)} scenes, "
                          f"{len(lst_vals)} px, mean={mean_v:.1f}C")

        if s3_monthly_lst:
            coarse_source_used = "S3"
            print(f"\nS3 SLSTR: {len(s3_monthly_lst)} months with data.")
        else:
            print("\nNo S3 data found — falling back to MODIS.")

    except Exception as e:
        print(f"Planetary Computer error: {e}")
        print("Falling back to MODIS.")

# ─── MODIS FALLBACK ───
if coarse_source_used is None:
    if THERMAL_SOURCE == "S3":
        print("S3 unavailable — MODIS fallback.")
    mt = (ee.ImageCollection('MODIS/061/MOD11A1')
          .filterBounds(aoi)
          .filterDate(f'{YEAR_START}-01-01', f'{YEAR_END}-12-31')
          .map(compute_lst_modis))
    ma = (ee.ImageCollection('MODIS/061/MYD11A1')
          .filterBounds(aoi)
          .filterDate(f'{YEAR_START}-01-01', f'{YEAR_END}-12-31')
          .map(compute_lst_modis))
    modis_col = mt.merge(ma).sort('system:time_start').map(add_time_props)
    coarse_source_used = "MODIS"
    print(f"MODIS: {modis_col.size().getInfo()} scenes")

print(f"\nThermal source: {coarse_source_used}")

Querying Sentinel-3 SLSTR LST on Planetary Computer...

No S3 data found — falling back to MODIS.
S3 unavailable — MODIS fallback.
MODIS: 5084 scenes

Thermal source: MODIS


In [10]:
# DEBUG: Check what Planetary Computer actually has for S3 SLSTR LST
import planetary_computer as pc
from pystac_client import Client as STACClient

catalog = STACClient.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace
)

# Step 1: Does the collection exist?
try:
    col = catalog.get_collection("sentinel-3-slstr-lst-l2-netcdf")
    print(f"Collection found: {col.id}")
    print(f"  Title: {col.title}")
    print(f"  Temporal: {col.extent.temporal.intervals}")
    print(f"  Spatial: {col.extent.spatial.bboxes}")
except Exception as e:
    print(f"Collection NOT found: {e}")
    print("\nSearching for anything with 'sentinel-3'...")
    for c in catalog.get_collections():
        if 'sentinel-3' in c.id.lower():
            print(f"  Found: {c.id} — {c.title}")

# Step 2: Try a broad search (no bbox) for recent data
print("\n--- Broad search (no bbox, last 3 months) ---")
try:
    search = catalog.search(
        collections=["sentinel-3-slstr-lst-l2-netcdf"],
        datetime="2025-01-01/2025-03-31",
        max_items=3
    )
    items = list(search.items())
    print(f"Found {len(items)} items globally")
    for item in items[:3]:
        print(f"  {item.id} | bbox: {item.bbox}")
        print(f"  Assets: {list(item.assets.keys())}")
except Exception as e:
    print(f"Search failed: {e}")

# Step 3: Try with the Manila bbox specifically
print("\n--- Manila bbox search ---")
try:
    search = catalog.search(
        collections=["sentinel-3-slstr-lst-l2-netcdf"],
        bbox=[120.9, 14.5, 121.1, 14.7],
        datetime="2024-01-01/2024-12-31",
        max_items=5
    )
    items = list(search.items())
    print(f"Found {len(items)} items over Manila")
except Exception as e:
    print(f"Manila search failed: {e}")

Collection found: sentinel-3-slstr-lst-l2-netcdf
  Title: Sentinel-3 Land Surface Temperature
  Temporal: [[datetime.datetime(2016, 4, 19, 1, 35, 17, 188500, tzinfo=tzutc()), None]]
  Spatial: [[-180, -90, 180, 90]]

--- Broad search (no bbox, last 3 months) ---
Found 3 items globally
  S3A_SL_2_LST_20250331T235731_20250401T000031_0179_124_173_1800 | bbox: [-180.0, 59.3242, 180.0, 73.3564]
  Assets: ['lst-in', 'slstr-met-tx', 'safe-manifest', 'slstr-time-in', 'slstr-flags-in', 'lst-ancillary-ds', 'slstr-indices-in', 'slstr-geodetic-in', 'slstr-geodetic-tx', 'slstr-geometry-tn', 'slstr-cartesian-in', 'slstr-cartesian-tx']
  S3A_SL_2_LST_20250331T235431_20250331T235731_0179_124_173_1620 | bbox: [-180.0, 68.5132, 180.0, 83.7799]
  Assets: ['lst-in', 'slstr-met-tx', 'safe-manifest', 'slstr-time-in', 'slstr-flags-in', 'lst-ancillary-ds', 'slstr-indices-in', 'slstr-geodetic-in', 'slstr-geodetic-tx', 'slstr-geometry-tn', 'slstr-cartesian-in', 'slstr-cartesian-tx']
  S3A_SL_2_LST_20250331T2351

In [ ]:
# 4C. UNIFIED COARSE INTERFACE
def get_coarse_composite(months_filter=None, year=None, month=None):
    if coarse_source_used == "S3":
        if year and month and (year,month) in s3_monthly_lst:
            return s3_monthly_lst[(year,month)]
        elif months_filter:
            imgs = [i for (y,m),i in s3_monthly_lst.items() if m in months_filter]
            if imgs: return ee.ImageCollection(imgs).mean().rename('LST_coarse').clip(aoi_buffer)
        if s3_monthly_lst:
            return ee.ImageCollection(list(s3_monthly_lst.values())).mean().rename('LST_coarse').clip(aoi_buffer)
        return ee.Image.constant(30).rename('LST_coarse').clip(aoi_buffer).float()
    else:
        if year and month:
            s = f'{year}-{month:02d}-01'
            return modis_col.filterDate(s, ee.Date(s).advance(1,'month')).select('LST_coarse').median().clip(aoi_buffer)
        elif months_filter:
            return filter_by_months(modis_col, months_filter).select('LST_coarse').median().clip(aoi_buffer)
        return modis_col.select('LST_coarse').median().clip(aoi_buffer)

tv = get_coarse_composite(months_filter=DRY_MONTHS).reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi,scale=1000,maxPixels=1e9).get('LST_coarse').getInfo()
print(f"Coarse interface OK — dry mean: {tv:.1f}C ({coarse_source_used})")


## 4D. Resolution Stack

In [ ]:
# 4D. STACK
cl = "Sentinel-3 SLSTR" if coarse_source_used=="S3" else "MODIS"
print(f"Tier 1 (30m): LST,NDVI,NDBI,MNDWI — Landsat 8+9")
print(f"Tier 2 (10m): LandCover,BuiltUp,Buildings — ESA/GHSL/OpenBldgs")
print(f"Tier 3 (100m): Population — WorldPop")
print(f"Tier 4 (1km): LST_coarse — {cl}")
print(f"Fused (30m): LST_fused — {cl}->30m regression")

## 5. Composites

In [ ]:
# 5. COMPOSITES
dry_landsat = filter_by_months(landsat_col,DRY_MONTHS).select(["LST","NDVI","NDBI","MNDWI"]).median().clip(aoi_buffer)
wet_landsat = filter_by_months(landsat_col,WET_MONTHS).select(["LST","NDVI","NDBI","MNDWI"]).median().clip(aoi_buffer)
ann_landsat = landsat_col.select(["LST","NDVI","NDBI","MNDWI"]).median().clip(aoi_buffer)
dry_coarse = get_coarse_composite(months_filter=DRY_MONTHS)
wet_coarse = get_coarse_composite(months_filter=WET_MONTHS)
ann_coarse = get_coarse_composite()
print(f"Composites built (Landsat 30m + {coarse_source_used})")

## 5B. LST Fusion (coarse → 30 m)

In [ ]:
# 5B. FUSION FUNCTION
def fuse_lst(lcomp, ccomp, geom, label=""):
    p30 = lcomp.select(['NDVI','NDBI','MNDWI']).clip(geom)
    p1k = p30.reduceResolution(reducer=ee.Reducer.mean(),maxPixels=2048).reproject(crs='EPSG:4326',scale=1000)
    clst = ccomp.select('LST_coarse').clip(geom)
    ts = clst.addBands(p1k)
    tr = ts.sample(region=geom, scale=1000, numPixels=500, seed=42, geometries=False)
    n = tr.size().getInfo()
    if n < 10:
        print(f"  {label}: {n} samples — Landsat only")
        return lcomp.select('LST').rename('LST_fused')
    tr = tr.map(lambda f: f.set('constant',1))
    reg = tr.reduceColumns(reducer=ee.Reducer.linearRegression(numX=4,numY=1),
                           selectors=['constant','NDVI','NDBI','MNDWI','LST_coarse'])
    co = ee.Array(reg.get('coefficients')).project([0]).toList()
    b0=ee.Number(co.get(0)); b1=ee.Number(co.get(1)); b2=ee.Number(co.get(2)); b3=ee.Number(co.get(3))
    fused = p30.select('NDVI').multiply(b1).add(p30.select('NDBI').multiply(b2)).add(p30.select('MNDWI').multiply(b3)).add(b0).rename('LST_fused')
    pred1k = p1k.select('NDVI').multiply(b1).add(p1k.select('NDBI').multiply(b2)).add(p1k.select('MNDWI').multiply(b3)).add(b0)
    res = clst.subtract(pred1k).resample('bilinear').reproject(crs='EPSG:4326',scale=30)
    fused = fused.add(res).rename('LST_fused')
    print(f"  {label}: n={n}, b0={b0.getInfo():.2f}, bNDVI={b1.getInfo():.3f}, bNDBI={b2.getInfo():.3f}, bMNDWI={b3.getInfo():.3f}")
    return fused
print("Fusion function ready.")


In [ ]:
# 5C. RUN FUSION
print(f"Fusing ({coarse_source_used} -> 30m)...")
fused_dry = fuse_lst(dry_landsat, dry_coarse, aoi_buffer, "Dry")
fused_wet = fuse_lst(wet_landsat, wet_coarse, aoi_buffer, "Wet")
fused_ann = fuse_lst(ann_landsat, ann_coarse, aoi_buffer, "Annual")
dry_composite = dry_landsat.addBands(fused_dry)
wet_composite = wet_landsat.addBands(fused_wet)
annual_composite = ann_landsat.addBands(fused_ann)
print("Done: LST + LST_fused + NDVI + NDBI + MNDWI")

## 5D. Monthly Time-Series

In [ ]:
# 5D. MONTHLY
monthly_results = []
print(f"Monthly fusion ({coarse_source_used})...")
for yr in range(YEAR_START, YEAR_END+1):
    for mo in range(1,13):
        s = f'{yr}-{mo:02d}-01'
        ed = ee.Date(s).advance(1,'month')
        lmo = landsat_col.filterDate(s, ed).select(['LST','NDVI','NDBI','MNDWI'])
        ln = lmo.size().getInfo()
        cmo = get_coarse_composite(year=yr, month=mo)
        cv = cmo.reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi,scale=1000,maxPixels=1e9).get('LST_coarse').getInfo()
        if ln==0 and cv is None: continue
        if ln>=1 and cv is not None:
            fused = fuse_lst(lmo.median().clip(aoi_buffer), cmo, aoi_buffer, f"{yr}-{mo:02d}")
            method = f"Fused ({coarse_source_used})"
        elif ln>=1:
            fused = lmo.select('LST').median().clip(aoi_buffer).rename('LST_fused')
            method = "Landsat only"
        else:
            fused = cmo.select('LST_coarse').resample('bilinear').reproject(crs='EPSG:4326',scale=30).rename('LST_fused')
            method = f"{coarse_source_used} resampled"
        mv = fused.reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi,scale=30,maxPixels=1e9).get('LST_fused').getInfo()
        monthly_results.append({'Year':yr,'Month':mo,'Date':f'{yr}-{mo:02d}','L_scenes':ln,'Method':method,'Mean_LST':round(mv,2) if mv else None})
        if mv: print(f"  {yr}-{mo:02d} L={ln} {method} {mv:.1f}C")
monthly_df = pd.DataFrame(monthly_results)
print(f"\n{len(monthly_df)} months.")


In [ ]:
# 5E. PLOT
fig,ax=plt.subplots(figsize=(14,5))
bc={"Fused (S3)":"#d73027","Fused (MODIS)":"#d73027","Landsat only":"#4575b4","S3 resampled":"#fdae61","MODIS resampled":"#fdae61"}
for method,grp in monthly_df.groupby("Method"):
    ax.plot(grp["Date"],grp["Mean_LST"],"o-",color=bc.get(method,"gray"),label=method,ms=4,lw=1)
ax.set_xlabel("Month");ax.set_ylabel("LST (C)");ax.set_title(f"{PROJECT_NAME} — Monthly LST")
ax.legend();ax.set_xticks(ax.get_xticks()[::3]);plt.xticks(rotation=45)
plt.tight_layout();plt.savefig("monthly_lst.png",dpi=150,bbox_inches="tight");plt.show()

## 5F. Fusion Validation

In [ ]:
# 5F. VALIDATION
vs = annual_composite.select("LST").rename("L").addBands(annual_composite.select("LST_fused").rename("F"))
vd = pd.DataFrame([f["properties"] for f in vs.sample(region=aoi,scale=30,numPixels=1000,seed=42,geometries=False).getInfo()["features"]]).dropna()
fig,axes=plt.subplots(1,3,figsize=(16,5))
axes[0].scatter(vd["L"],vd["F"],alpha=0.3,s=8,c="#d73027");axes[0].plot([vd.min().min(),vd.max().max()],[vd.min().min(),vd.max().max()],"k--")
axes[0].set_xlabel("Landsat");axes[0].set_ylabel("Fused");axes[0].set_title("Scatter")
axes[1].hist(vd["L"],bins=40,alpha=0.6,color="#4575b4",label="Landsat");axes[1].hist(vd["F"],bins=40,alpha=0.6,color="#d73027",label="Fused");axes[1].legend();axes[1].set_title("Distribution")
d=vd["F"]-vd["L"];axes[2].hist(d,bins=40,color="#fdae61",alpha=0.7);axes[2].axvline(0,color="k",ls="--")
axes[2].set_title(f"Residual (mean={d.mean():.2f})");plt.tight_layout();plt.savefig("fusion_val.png",dpi=150);plt.show()

## 6. UHI Intensity

In [ ]:
# 6. UHI
def compute_uhi(comp, band, label):
    lst=comp.select(band)
    ref=ee.Number(lst.reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi_ring,scale=30,maxPixels=1e9).get(band))
    print(f"  {label} ref:{ref.getInfo():.2f}C")
    return lst.subtract(ref).rename("UHI_intensity")
print("Fused UHI:")
uhi_dry=compute_uhi(dry_composite,"LST_fused","Dry")
uhi_wet=compute_uhi(wet_composite,"LST_fused","Wet")
uhi_annual=compute_uhi(annual_composite,"LST_fused","Annual")
print("\nLandsat UHI:")
uhi_dry_l=compute_uhi(dry_composite,"LST","Dry")
uhi_wet_l=compute_uhi(wet_composite,"LST","Wet")
uhi_annual_l=compute_uhi(annual_composite,"LST","Annual")

## 7. Maps

In [ ]:
# 7. MAP
lv={"min":25,"max":45,"palette":["313695","4575b4","74add1","abd9e9","e0f3f8","ffffbf","fee090","fdae61","f46d43","d73027","a50026"]}
uv={"min":-3,"max":6,"palette":["2166ac","67a9cf","d1e5f0","fddbc7","ef8a62","b2182b"]}
nv={"min":-0.1,"max":0.6,"palette":["FFFFFF","CE7E45","DF923D","F1B555","FCD163","99B718","74A901","66A000","529400","3E8601","207401","056201"]}
m2=make_map(center_lat,center_lon,14)
add_ee_layer(m2,dry_composite.select("LST_fused").clip(aoi),lv,"Fused LST Dry")
add_ee_layer(m2,wet_composite.select("LST_fused").clip(aoi),lv,"Fused LST Wet",False)
add_ee_layer(m2,annual_composite.select("LST").clip(aoi),lv,"Landsat LST",False)
add_ee_layer(m2,uhi_dry.clip(aoi),uv,"UHI Dry")
add_ee_layer(m2,uhi_wet.clip(aoi),uv,"UHI Wet",False)
add_ee_layer(m2,annual_composite.select("NDVI").clip(aoi),nv,"NDVI",False)
folium.LayerControl().add_to(m2)
m2

---
## 8. Ancillary Data

In [ ]:
# 8. ANCILLARY
pop=(ee.ImageCollection("WorldPop/GP/100m/pop").filterBounds(aoi).sort("system:time_start",False).first().select("population").clip(aoi))
print(f"Pop: WorldPop {ee.Date(pop.get('system:time_start')).format('YYYY').getInfo()}")
ghsl=ee.Image("JRC/GHSL/P2023A/GHS_BUILT_S/2020").select("built_surface").clip(aoi);print("GHSL")
esa_lc=ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi);print("ESA WorldCover")
health_dist=None
try:
    ha=ee.FeatureCollection("projects/sat-io/open-datasets/health-site-node").filterBounds(aoi).merge(ee.FeatureCollection("projects/sat-io/open-datasets/health-site-way").filterBounds(aoi))
    hc=ha.size().getInfo()
    if hc>0: health_dist=ha.distance(5000).clip(aoi).rename("health_dist");print(f"Healthsites:{hc}")
    else: raise Exception("0")
except: health_dist=ee.Image.constant(0).rename("health_dist").clip(aoi).float();print("Health: placeholder")
try:
    ob=ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons").filterBounds(aoi)
    bldg_raster=ob.reduceToImage(properties=["confidence"],reducer=ee.Reducer.count()).rename("building_count").clip(aoi)
    print(f"OpenBldgs:{ob.size().getInfo()}")
except: bldg_raster=ghsl.rename("building_count");print("Bldgs: GHSL fallback")

---
## 9. Statistics

In [ ]:
# 9A. ZONAL
def zs(img,band,geom,sc=30):
    return img.select(band).reduceRegion(reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(),sharedInputs=True).combine(ee.Reducer.stdDev(),sharedInputs=True),geometry=geom,scale=sc,maxPixels=1e9).getInfo()
rows=[]
for sn,comp in [("Ann",annual_composite),("Dry",dry_composite),("Wet",wet_composite)]:
    for b,lb in [("LST_fused","Fused"),("LST","Landsat")]:
        s=zs(comp,b,aoi)
        rows.append({"Season":sn,"Src":lb,"Mean":round(s.get(f"{b}_mean",0),2),"Min":round(s.get(f"{b}_min",0),2),"Max":round(s.get(f"{b}_max",0),2),"Std":round(s.get(f"{b}_stdDev",0),2)})
stats_df=pd.DataFrame(rows);print(stats_df.to_string(index=False))

In [ ]:
# 9B+C. SAMPLING & CORRELATION
stk=annual_composite.select("LST_fused").rename("LST").addBands(annual_composite.select("NDVI")).addBands(annual_composite.select("NDBI")).addBands(annual_composite.select("MNDWI")).addBands(pop.rename("POP")).addBands(ghsl.rename("BUILT"))
df=pd.DataFrame([f["properties"] for f in stk.sample(region=aoi,scale=30,numPixels=2000,seed=42,geometries=False).getInfo()["features"]]).dropna()
print(f"{len(df)} pixels")
fig,axes=plt.subplots(2,3,figsize=(16,10));fig.suptitle(f"{PROJECT_NAME} — Correlation",fontsize=14)
corr=df[["LST","NDVI","NDBI","MNDWI","POP","BUILT"]].corr()
sns.heatmap(corr,annot=True,cmap="RdBu_r",center=0,ax=axes[0,0],fmt=".2f")
for ax,col,c,t in [(axes[0,1],"NDVI","green","NDVI"),(axes[0,2],"NDBI","orange","NDBI"),(axes[1,0],"POP","purple","Pop"),(axes[1,1],"BUILT","brown","Built")]:
    ax.scatter(df[col],df["LST"],alpha=0.3,s=5,c=c);ax.set_xlabel(col);ax.set_ylabel("LST");ax.set_title(f"LST vs {t}")
axes[1,2].hist(df["LST"],bins=40,color="#d73027",alpha=0.7);axes[1,2].set_title("LST Dist")
plt.tight_layout();plt.savefig("correlation.png",dpi=150);plt.show()

## 10. Trends

In [ ]:
# 10. TRENDS
fig,axes=plt.subplots(2,1,figsize=(14,9),gridspec_kw={"height_ratios":[2,1]})
v=monthly_df.dropna(subset=["Mean_LST"])
axes[0].plot(v["Date"],v["Mean_LST"],"o-",color="#d73027",ms=3,lw=1)
axes[0].set_ylabel("LST (C)");axes[0].set_title(f"{PROJECT_NAME} — Monthly")
axes[0].set_xticks(axes[0].get_xticks()[::6]);axes[0].tick_params(axis="x",rotation=45)
y=v.groupby("Year").agg(M=("Mean_LST","mean"),N=("Mean_LST","count")).reset_index()
axes[1].bar(y["Year"].astype(str),y["M"],color="#fc8d59")
for _,r in y.iterrows(): axes[1].annotate(f"{r['M']:.1f}",(str(int(r["Year"])),r["M"]),ha="center",va="bottom",fontsize=9)
axes[1].set_ylabel("LST");axes[1].set_title("Yearly")
plt.tight_layout();plt.savefig("trends.png",dpi=150);plt.show()

---
## 11. Vulnerability Index

In [ ]:
# 11. VULNERABILITY
lst_norm=min_max_normalize(annual_composite,aoi,"LST_fused").rename("LST_norm")
pop_norm=min_max_normalize(pop,aoi).rename("POP_norm")
built_norm=min_max_normalize(ghsl,aoi).rename("BUILT_norm")
ndvi_inv=ee.Image.constant(1).subtract(min_max_normalize(annual_composite,aoi,"NDVI")).rename("NDVI_inv")
health_norm=min_max_normalize(health_dist,aoi).rename("HEALTH_norm")
vulnerability=(lst_norm.multiply(W_LST).add(pop_norm.multiply(W_POP)).add(built_norm.multiply(W_BUILD)).add(ndvi_inv.multiply(W_NDVI)).add(health_norm.multiply(W_HEALTH))).rename("vulnerability").clip(aoi)
print(f"Vulnerability computed. W: LST={W_LST} POP={W_POP} BUILD={W_BUILD} NDVI={W_NDVI} HEALTH={W_HEALTH}")

In [ ]:
# 12. VULN MAP
vv={"min":0,"max":1,"palette":["1a9850","91cf60","d9ef8b","ffffbf","fee08b","fc8d59","d73027"]}
m3=make_map(center_lat,center_lon,14)
add_ee_layer(m3,vulnerability,vv,"Vulnerability")
add_ee_layer(m3,lst_norm.clip(aoi),{"min":0,"max":1,"palette":["blue","yellow","red"]},"LST",False)
add_ee_layer(m3,pop_norm.clip(aoi),{"min":0,"max":1,"palette":["white","purple"]},"Pop",False)
add_ee_layer(m3,ndvi_inv.clip(aoi),{"min":0,"max":1,"palette":["green","white","brown"]},"VegDeficit",False)
folium.LayerControl().add_to(m3)
m3

## 12. Barangay Ranking

In [ ]:
# 13. BARANGAY
try:
    bg=ee.FeatureCollection("projects/sat-io/open-datasets/geoboundaries/CGAZ_ADM4").filterBounds(aoi)
    if bg.size().getInfo()==0: raise Exception("0")
    bv=vulnerability.reduceRegions(collection=bg,reducer=ee.Reducer.mean().setOutputs(["vm"]),scale=30)
    bl=annual_composite.select("LST_fused").reduceRegions(collection=bg,reducer=ee.Reducer.mean().setOutputs(["lm"]),scale=30)
    rk=[{"Barangay":v["properties"].get("shapeName","?"),"Vuln":round(v["properties"].get("vm",0),3),"LST":round(l["properties"].get("lm",0),2)} for v,l in zip(bv.getInfo()["features"],bl.getInfo()["features"])]
    rank_df=pd.DataFrame(rk).sort_values("Vuln",ascending=False);print(rank_df.to_string(index=False))
except Exception as e: print(f"Barangay N/A: {e}")

---
## 13. Export

In [ ]:
# 14. EXPORT
ep=dict(region=aoi,scale=30,maxPixels=1e9,crs="EPSG:4326")
for img,nm in [(vulnerability.toFloat(),"vulnerability"),(annual_composite.select("LST_fused").toFloat(),"LST_fused"),(annual_composite.select("LST").toFloat(),"LST_landsat"),(uhi_dry.toFloat(),"UHI_dry"),(uhi_annual.toFloat(),"UHI_annual")]:
    ee.batch.Export.image.toDrive(image=img,description=f"{PROJECT_NAME}_{nm}",folder=PROJECT_NAME,**ep).start();print(f"Export: {nm}")
print("https://code.earthengine.google.com/tasks")
stats_df.to_csv("stats.csv",index=False);monthly_df.to_csv("monthly.csv",index=False)
if "rank_df" in dir(): rank_df.to_csv("ranking.csv",index=False)
print("CSVs saved.")

---
## 14. Cooling Sites

In [ ]:
# 15. COOLING
cool=esa_lc.eq(10).Or(esa_lc.eq(20)).Or(esa_lc.eq(30)).selfMask().rename("cool")
vp75=vulnerability.reduceRegion(reducer=ee.Reducer.percentile([75]),geometry=aoi,scale=30,maxPixels=1e9).get("vulnerability")
hv=vulnerability.gte(ee.Number(vp75)).selfMask()
m4=make_map(center_lat,center_lon,14)
add_ee_layer(m4,vulnerability,vv,"Vulnerability")
add_ee_layer(m4,hv.clip(aoi),{"min":0,"max":1,"palette":["red"]},"High Vuln")
add_ee_layer(m4,cool.clip(aoi),{"min":0,"max":1,"palette":["00ff00"]},"Cool Spaces",False)
folium.LayerControl().add_to(m4)
m4

---
## Summary (v3)

| Output | Description |
|--------|-------------|
| Landsat 8+9 | 30 m, ~8-day combined |
| Sentinel-3 SLSTR / MODIS | 1 km coarse thermal (configurable) |
| Fused LST | Regression downscaling → 30 m |
| Monthly time-series | Every month, fused |
| UHI intensity | Fused + Landsat comparison |
| Vulnerability index | LST + pop + built-up + veg + health |
| Exports | GeoTIFFs + CSVs |

**Switch source**: `THERMAL_SOURCE = "S3"` or `"MODIS"` in config.  
**New AOI**: Change `AOI_BBOX`, re-run all.

---
*MSF · Z_GIS Salzburg*
